# Notebook Name: 01_etl_and_data_cleaning.ipynb

## Objectives
* Load raw patient encounter data (`diabetic_data.csv`) and mapping metadata (`IDS_mapping.csv`).
* Convert missing value indicators (`?`) to standard missing values (`NaN`).
* Derive binary classification target variable (`readmitted_30`) for 30-day hospital readmissions.
* Filter out expired and hospice-discharged patients to avoid clinical data leakage.
* Deduplicate encounters by retaining each patient's initial recorded encounter.
* Clean missing values and drop features with zero variance or high missingness.
* Export the cleaned dataset to `data/processed/cleaned_diabetic_data.csv`.

## Inputs
* `data/raw/diabetic_data.csv`: Primary dataset containing 101,766 patient encounters across 50 features.
* `data/raw/IDS_mapping.csv`: ID mapping table for admission types, discharge dispositions, and admission sources.

## Outputs
* `data/processed/cleaned_diabetic_data.csv`: Standardized, deduplicated, and leak-free dataset ready for EDA and machine learning.

## Additional Comments
* Discharge disposition IDs `11`, `13`, `14`, `19`, `20`, and `21` refer to deceased patients or hospice care. Removing these prevents the machine learning model from predicting readmission on non-eligible patient records (clinical data leakage).

## The Pipeline
1. **Executive Overview**
2. **Extract & Load**
3. **Data Structure & Quality Inspection**
4. **Data Cleaning & Transformation**
5. **Data Export & Summary**

## 1. Executive Overview
This notebook executes the initial Extraction, Transformation, and Loading (ETL) phase of the data pipeline. It cleans raw hospital encounter data from 130 US hospitals, resolves relational ID mappings, removes post-discharge clinical leakage, and formats a clean binary target feature (`readmitted_30`) for risk modeling.

## 2. Extract & Load

To initiate data pipeline, I begin by importing the required core Python libraries (`pandas` and `numpy`) for data extraction, manipulation, and array operations. 

I then extract the primary dataset (`diabetic_data.csv`) directly into a Pandas DataFrame named `df`, alongside the auxiliary lookup table (`IDS_mapping.csv`). The mapping table will later serve to decode categorical numeric identifiers such as admission types and discharge dispositions.

Finally, I perform an immediate validation check by printing the dataset dimensions (`df.shape`) and previewing the initial rows (`df.head()`) and trailing rows (`df.tail()`) to confirm accurate file reading and structural integrity.

In [3]:
import pandas as pd
import numpy as np

# 2. Extract & Load
df = pd.read_csv('../data/raw/diabetic_data.csv')
ids_mapping = pd.read_csv('../data/raw/IDS_mapping.csv')

print(f"Raw dataset successfully loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
print("--- RAW DATASET HEAD (First 5 Rows) ---")
display(df.head(5))

print("\n--- RAW DATASET TAIL (Last 5 Rows) ---")
display(df.tail(5))

Raw dataset successfully loaded with 101766 rows and 50 columns.
--- RAW DATASET HEAD (First 5 Rows) ---


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO



--- RAW DATASET TAIL (Last 5 Rows) ---


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO
101765,443867222,175429310,Caucasian,Male,[70-80),?,1,1,7,6,...,No,No,No,No,No,No,No,No,No,NO


## 3. Data Structure & Quality Inspection

Prior to performing data transformations, I conducted a data quality audit to understand feature types and inspect structural anomalies.

In this stage, I:
1. **Inspected Data Types (`df.dtypes`):** Verify numeric features versus categorical object types.
2. **Examined Explicit Missing Values (`NaN`):** Check for missing entries recognized natively by Pandas.
3. **Detected Implicit Placeholder Values (`?`):** In clinical database exports, missing information is frequently coded as specific characters rather than empty fields. I inspect for question mark placeholders (`?`) across all features to quantify true missingness before cleaning.

In [4]:
# 1. Summary of column data types and basic non-null counts
print("=== DATAFRAME INFORMATION & DATA TYPES ===")
print(df.info())

# 2. Check for native nulls
native_nulls = df.isnull().sum()
print("\n=== NATIVE NULL VALUES PER COLUMN ===")
print(native_nulls[native_nulls > 0] if native_nulls.sum() > 0 else "No native NaN values found.")

# 3. Detect non-standard missing value indicators ('?')
question_mark_counts = (df == '?').sum()
missing_placeholders = question_mark_counts[question_mark_counts > 0]

print("\n=== IMPLICIT MISSING VALUE INDICATORS ('?') DETECTED ===")
print(missing_placeholders)

=== DATAFRAME INFORMATION & DATA TYPES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures  

## 4. Data Cleaning & Transformation

 I executed a structured cleaning pipeline designed to optimize the dataset for machine learning while eliminating clinical data leakage:

1. **Standardize Missing Values:** Convert all implicit `?` placeholders into standard `np.nan` missing value types.
2. **Derive Target Feature (`readmitted_30`):** Re-code the multi-class target `readmitted` into a binary target (`1` if readmitted in $<30$ days, `0` otherwise).
3. **Prevent Clinical Data Leakage:** Remove records where `discharge_disposition_id` corresponds to deceased or hospice-transferred patients (IDs `11, 13, 14, 19, 20, 21`), as these patients are ineligible for readmission.
4. **Ensure Statistical Independence:** Retain only the first hospital encounter per patient (`patient_nbr`) to maintain independent observations.
5. **Drop Non-Informative Columns:** Exclude features with high missingness rates (`weight`, `payer_code`) or zero variance (`examide`, `citoglipton`).
6. **Impute Categorical Features:** Fill remaining `NaN` entries in categorical fields with an explicit `'Unknown'` label.

In [6]:
# 1. Standardize missing indicators
clean_df = df.replace('?', np.nan)

# 2. Derive binary target variable
clean_df['readmitted_30'] = (clean_df['readmitted'] == '<30').astype(int)

# 3. Prevent clinical leakage (filter out deceased/hospice IDs)
expired_hospice_ids = [11, 13, 14, 19, 20, 21]
filtered_df = clean_df[~clean_df['discharge_disposition_id'].isin(expired_hospice_ids)].copy()

# 4. Deduplicate to keep initial patient encounter
dedup_df = filtered_df.drop_duplicates(subset=['patient_nbr'], keep='first').copy()

# 5. Remove high missingness / constant columns
cols_to_drop = ['weight', 'payer_code', 'examide', 'citoglipton']
dedup_df.drop(columns=[col for col in cols_to_drop if col in dedup_df.columns], inplace=True)

# 6. Impute remaining missing categorical fields
impute_map = {
    'medical_specialty': 'Unknown',
    'race': 'Unknown',
    'diag_1': 'Unknown',
    'diag_2': 'Unknown',
    'diag_3': 'Unknown'
}
dedup_df.fillna(value=impute_map, inplace=True)

print(f"Data cleaning pipeline completed successfully.")
print(f"Original Shape: {df.shape} | Cleaned Shape: {dedup_df.shape}\n")

# Preview cleaned DataFrame state
print("--- CLEANED DATAFRAME HEAD ---")
display(dedup_df.head(5))

print("\n--- CLEANED DATAFRAME TAIL ---")
display(dedup_df.tail(5))

Data cleaning pipeline completed successfully.
Original Shape: (101766, 50) | Cleaned Shape: (69990, 47)

--- CLEANED DATAFRAME HEAD ---


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,No,Yes,NO,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,Steady,No,No,No,No,No,Ch,Yes,NO,0



--- CLEANED DATAFRAME TAIL ---


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30
101754,443842016,183087545,Caucasian,Female,[70-80),1,1,7,9,Unknown,...,Steady,No,No,No,No,No,Ch,Yes,>30,0
101755,443842022,188574944,Other,Female,[40-50),1,1,7,14,Unknown,...,Up,No,No,No,No,No,Ch,Yes,>30,0
101756,443842070,140199494,Other,Female,[60-70),1,1,7,2,Unknown,...,Steady,No,No,No,No,No,No,Yes,>30,0
101758,443842340,120975314,Caucasian,Female,[80-90),1,1,7,5,Unknown,...,Up,No,No,No,No,No,Ch,Yes,NO,0
101765,443867222,175429310,Caucasian,Male,[70-80),1,1,7,6,Unknown,...,No,No,No,No,No,No,No,No,NO,0


## 5. Data Export & Summary

In the final step of the ETL pipeline, I saved the cleaned DataFrame as a CSV file to `data/processed/cleaned_diabetic_data.csv`.

I then re-loaded the processed file to verify its shape, column integrity, and target distribution (`readmitted_30`), ensuring it is completely prepared for Notebook 02 (EDA and Descriptive Statistics).

In [7]:
# Export cleaned dataset
output_path = '../data/processed/cleaned_diabetic_data.csv'
dedup_df.to_csv(output_path, index=False)

print(f"Dataset successfully exported to: {output_path}")

# Reload and validate exported file
verify_df = pd.read_csv(output_path)
print(f"Verified Processed File Shape: {verify_df.shape[0]} rows x {verify_df.shape[1]} columns")

print("\n=== TARGET CLASS DISTRIBUTION ('readmitted_30') ===")
print(verify_df['readmitted_30'].value_counts(normalize=True).round(4) * 100)

Dataset successfully exported to: ../data/processed/cleaned_diabetic_data.csv
Verified Processed File Shape: 69990 rows x 47 columns

=== TARGET CLASS DISTRIBUTION ('readmitted_30') ===
readmitted_30
0    91.02
1     8.98
Name: proportion, dtype: float64


## 6. Preparation Summary

This initial ETL notebook successfully prepares the raw patient dataset for exploratory analysis and predictive modeling:

* **Ingestion:** I extracted 101,766 raw encounters across 50 features from `data/raw/diabetic_data.csv` and relational ID descriptions from `data/raw/IDS_mapping.csv`.
* **Clinical Leakage Prevention:** I filtered out records with discharge disposition IDs corresponding to expired or hospice care cases (IDs `11`, `13`, `14`, `19`, `20`, `21`), ensuring model evaluation focuses strictly on patients eligible for readmission.
* **Deduplication:** I isolated initial encounters per patient (`patient_nbr`), reducing duplicate record bias and securing observation independence across clean encounters.
* **Target Isolation:** I created `readmitted_30`, a binary classification target where `1` denotes readmission under 30 days and `0` denotes non-readmission.
* **Feature Refinement:** I dropped non-informative columns (`weight`, `payer_code`, `examide`, `citoglipton`) and standardized missing categorical variables with explicit `'Unknown'` labels.

## 7. Version Control Workflow

To maintain full alignment with software engineering best practices, I tracked and pushed changes using conventional commit standard tags:
```bash
# Stage the notebook and processed output dataset
git add notebooks/01_etl_and_data_cleaning.ipynb data/processed/cleaned_diabetic_data.csv

# Commit using descriptive conventional commit message
git commit -m "feat(etl): complete extraction, leakage prevention, deduplication, and dataset export"

# Push updates to main repository branch
git push origin main

## 8. Next Step
With data cleaning complete, the next step is Exploratory Data Analysis and Descriptive Statistics.

👉 **[Continue to Notebook 02: Exploratory Data Analysis & Descriptive Statistics](02_eda_and_descriptive_statistics.ipynb)**

## 9. Acknowledgements and Credits
**Data Source:** UCI Machine Learning Repository — Diabetes 130-US Hospitals for Years 1999-2008 Data Set. Provided by Strack et al. (2014) via Hindawi Publishing Corporation.

**Domain Context:** Guidelines on 30-day hospital readmissions metrics established under the Centers for Medicare & Medicaid Services (CMS) Hospital Readmissions Reduction Program (HRRP).

## 10. Sources and Layout Inspiration
**Code Institute Assessment Framework:** Template architecture adapted from the Code Institute Advanced Data Analytics and Machine Learning Capstone Handbook.

**Methodology:** Data cleaning pipelines designed following CRISP-DM (Cross-Industry Standard Process for Data Mining) industry practices.

## 11. Tooling and Assistance
**Development Environment:** Visual Studio Code with Jupyter Extension for interactive notebook development.

**Core Libraries:** Python, Pandas (data manipulation), NumPy (numerical array handling).

**AI Assistance:** Architectural advice and template code structuring, syntax debugging guided by Gemini AI, with all implementation, variable selection, and execution independently validated.